# Virtual Experiment Product Tour

This notebook is a researcher-facing exploratory tour of FungMod's public virtual-experiment API. It is not an empirical validation, calibration, or literature-comparison notebook. The example uses registry-backed records and explicit exploratory priors, then inspects the standard output tables that document assumptions, mechanism maturity, provenance, and limitations.

In [ ]:
import os
from pathlib import Path

from fungal_model import environment_grid, virtual_experiment

OUTPUT_ROOT = Path(os.environ.get("FUNGMOD_NOTEBOOK_OUTPUT_ROOT", "outputs/notebooks"))
OUTPUT_DIR = OUTPUT_ROOT / "10_virtual_experiment_product_tour"
REGISTRY = Path("data_registry/registry_index.yml")


Create a small virtual experiment from researcher-facing names. The runtime environment grid is metadata-only here unless explicit condition-specific parameter records or response laws exist, so the output tables should be used to inspect that limitation before interpreting environment differences.

In [ ]:
study = virtual_experiment(
    fungi="beta-glucosidase source",
    substrates="cellobiose substrate",
    environments=environment_grid(temperature_C=[30.0], ph=[5.0], oxygen="aerobic"),
    registry=REGISTRY,
)

preflight = study.preflight(mode="exploratory")
[(report.status, report.required_processes, report.suggested_experiments) for report in preflight]


Run a tiny exploratory simulation. `quicklook=False` keeps the smoke test fast; interactive users can set it to `True` to write plots.

In [ ]:
result = study.simulate(
    mode="exploratory",
    n_samples=1,
    seed=10,
    output_dir=OUTPUT_DIR,
    quicklook=False,
)

result.write_summary()
result.write_manifest()
sorted(Path(result.output_directory).glob("*.csv"))[:5]


Inspect mechanism and assumption rows before looking at curves. These tables are the guardrails that keep exploratory simulations separate from validation claims.

In [ ]:
mechanisms = result.mechanism_summary()
assumptions = result.assumption_summary()
limitations = result.limitations()

mechanisms[0], assumptions[:2], limitations[:2]


The standard tables can be loaded without rerunning the simulation. Here we inspect final metrics and sampled parameters, including source-class labels that show which inputs were exact records and which were exploratory priors.

In [ ]:
final_metrics = result.final_metrics()
sampled_parameters = result.sampled_parameters()

computed_metrics = [row for row in final_metrics if row["status"] == "computed"]
parameter_sources = [(row["symbol"], row["parameter_source_class"], row["allowed_use"]) for row in sampled_parameters]

computed_metrics[:3], parameter_sources
